# Transformers for Audio Tasks

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/Building_with_Deep_Learning/01-llms/tasks_audio.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/Building_with_Deep_Learning/01-llms"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


In [ ]:
# !pip install transformers==5.0.0 datasets==4.0.0 torch==2.10.0+cu128 accelerate==1.13.0

In [13]:
from rich import print

## 1. Audio Classification and Event Detection

Categorizing audio data into predefined classes (e.g., identifying a "dog bark," "glass breaking," or speaker intent/emotion).

Representative Models:

- **Audio Spectrogram Transformer (AST)** for environmental sounds.
- fine-tuned **Wav2Vec2** for speaker intent or language identification.

Example sub-task: classify bird based on an audio clip

![](../assets/birds_classificaition_from_sounds.png)

The `minds14` dataset is aboud classifying intent of customer speaking on the phone with a bank teller:

In [1]:
from datasets import load_dataset
from datasets import Audio

minds = load_dataset("PolyAI/minds14", name="en-AU", split="train")
minds = minds.cast_column("audio", Audio(sampling_rate=16_000))


README.md: 0.00B [00:00, ?B/s]

en-AU/train-00000-of-00001.parquet:   0%|          | 0.00/37.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/654 [00:00<?, ? examples/s]

To classify an audio recording into a set of classes, we can use the `audio-classification` pipeline from 🤗 Transformers. In our case, we need a model that's been fine-tuned for intent classification, and specifically on the MINDS-14 dataset. Luckily for us, the Hub has a model that does just that! Let's load it by using the `pipeline()` function:

In [2]:
from transformers import pipeline

classifier = pipeline(
    "audio-classification",
    model="anton-l/xtreme_s_xlsr_300m_minds14",
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

This pipeline expects the audio data as a NumPy array. All the preprocessing of the raw audio data will be conveniently handled for us by the pipeline. Let's pick an example to try it out:

In [3]:
example = minds[0]

If you recall the structure of the dataset, the raw audio data is stored in a NumPy array under `["audio"]["array"]`, let's pass it straight to the `classifier`:

In [14]:
output = classifier(example["audio"]["array"])
print(output)

[
    {'score': 0.9659619927406311, 'label': 'pay_bill'},
    {'score': 0.025758082047104836, 'label': 'freeze'},
    {'score': 0.0030119898729026318, 'label': 'card_issues'},
    {'score': 0.0018934155814349651, 'label': 'abroad'},
    {'score': 0.0008216770365834236, 'label': 'high_value_payment'},
    {'score': 0.000717991846613586, 'label': 'direct_debit'},
    {'score': 0.000390784814953804, 'label': 'latest_transactions'},
    {'score': 0.00033559236908331513, 'label': 'joint_account'},
    {'score': 0.00033539062133058906, 'label': 'balance'},
    {'score': 0.0003297204675618559, 'label': 'address'},
    {'score': 0.0001458501792512834, 'label': 'atm_limit'},
    {'score': 0.00014547427417710423, 'label': 'app_error'},
    {'score': 8.655788406031206e-05, 'label': 'cash_deposit'},
    {'score': 6.544432835653424e-05, 'label': 'business_loan'}
]

**Output:**
```out
[
    {"score": 0.9631525278091431, "label": "pay_bill"},
    {"score": 0.02819698303937912, "label": "freeze"},
    {"score": 0.0032787492964416742, "label": "card_issues"},
    {"score": 0.0019414445850998163, "label": "abroad"},
    {"score": 0.0008378693601116538, "label": "high_value_payment"},
]
```

The model is very confident that the caller intended to learn about paying their bill. Let's see what the actual label for this example is:

In [5]:
id2label = minds.features["intent_class"].int2str
id2label(example["intent_class"])

'pay_bill'

**Output:**
```out
"pay_bill"
```

Hooray! The predicted label was correct! Here we were lucky to find a model that can classify the exact labels that we need. A lot of the times, when dealing with a classification task, a pre-trained model's set of classes is not exactly the same as the classes you need the model to distinguish. In this case, you can fine-tune a pre-trained model to "calibrate" it to your exact set of class labels. We'll learn how to do this in the upcoming units. Now, let's take a look at another very common task in speech processing, _automatic speech recognition_.

## 2. Automatic Speech Recognition (ASR)

Converting spoken language into written text for transcription, voice commands, and accessibility.

Representative Models:

- **Whisper** (OpenAI) for robust multilingual transcription
- **SeamlessM4T** (Meta) for direct speech-to-text translation.


![](https://github.com/HassanAlgoz/AAI/blob/main/content/W5/M1/assets/asr.png?raw=1)

### Applications

- **Transcription of:**
  - **phone calls** for customer service and sales
  - **meetings** for note-taking and documentation
  - **lectures** for students and researchers
  - **podcasts** for SEO and accessibility
  - **interviews** for journalists and researchers
- **Dictation:** for people to speak their notes directly into text documents.
- **Voice Assistants:** AI-powered devices can answer questions, set reminders, and control smart homes (digitally connected).

We can fit a small model like [`openai/whisper-large-v3-turbo`](https://huggingface.co/openai/whisper-large-v3-turbo) (800 million parameters) and use it according to the model card [usage section](https://huggingface.co/openai/whisper-large-v3-turbo#usage).

In [6]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe_asr = pipeline(
    task="automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=torch_dtype,
    device=device,
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [7]:
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
sample = dataset[0]["audio"]

README.md:   0%|          | 0.00/480 [00:00<?, ?B/s]

clean/validation-00000-of-00001-91350812(…):   0%|          | 0.00/1.98M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

### ASR Parameters

In [8]:
transcription = pipe_asr(
    "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac",
    return_timestamps="word", # Returns precise start/end times for every word
    chunk_length_s=30,        # Splits long audio into 30-second segments for processing
    stride_length_s=5         # Adds 5 seconds of overlap between segments to prevent cutting words
)

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A 

In [15]:
print(transcription)

{
    'text': ' I have a dream that one day this nation will rise up and live out the true meaning of its creed.',
    'chunks': [
        {'text': ' I', 'timestamp': (0.0, 1.3)},
        {'text': ' have', 'timestamp': (1.3, 1.58)},
        {'text': ' a', 'timestamp': (1.58, 1.74)},
        {'text': ' dream', 'timestamp': (1.74, 2.06)},
        {'text': ' that', 'timestamp': (2.06, 3.78)},
        {'text': ' one', 'timestamp': (3.78, 4.04)},
        {'text': ' day', 'timestamp': (4.04, 4.34)},
        {'text': ' this', 'timestamp': (4.34, 6.54)},
        {'text': ' nation', 'timestamp': (6.54, 7.0)},
        {'text': ' will', 'timestamp': (7.0, 7.52)},
        {'text': ' rise', 'timestamp': (7.52, 8.06)},
        {'text': ' up', 'timestamp': (8.06, 8.54)},
        {'text': ' and', 'timestamp': (8.54, 10.28)},
        {'text': ' live', 'timestamp': (10.28, 10.46)},
        {'text': ' out', 'timestamp': (10.46, 10.7)},
        {'text': ' the', 'timestamp': (10.7, 10.9)},
        {'text': ' true', 'timestamp': (10.9, 11.14)},
        {'text': ' meaning', 'timestamp': (11.14, 11.48)},
        {'text': ' of', 'timestamp': (11.48, 11.7)},
        {'text': ' its', 'timestamp': (11.7, 11.9)},
        {'text': ' creed.', 'timestamp': (11.9, 12.5)}
    ]
}

### Fine-tuned ASR

> Example: [NAMAA-Space/EgypTalk-ASR-v2](https://huggingface.co/NAMAA-Space/EgypTalk-ASR-v2) trained on over 200 hours of high-quality, manually curated audio data collected and prepared by the NAMAA team. It is built upon NVIDIA’s FastConformer Hybrid Large architecture and fine-tuned for Egyptian Arabic, enabling highly accurate transcription in casual, formal, and mixed dialect settings.

## 3. Text-to-Speech (TTS)

Also known as: Voice Synthesis; Generating natural-sounding human speech from text, including voice cloning and emotional modulation for agents, audiobooks, and dubbing.

Representative Models:

- **Qwen3-TTS** or **CosyVoice2** for ultra-low latency, mobile-optimized streaming
- **VALL-E** (implementations) for zero-shot voice cloning.

> Example: [NAMAA-Saudi-TTS](https://huggingface.co/spaces/omarelshehy/NAMAA-Saudi-Voice) refined to generate natural Saudi dialect speech, targeting everyday conversational usage rather than Modern Standard Arabic (MSA).

### Applications

![](../assets/tts_applications.png)

1. **Customer Support (IVR)**: Operating automated phone menus and voice-based customer service bots to resolve user issues.
2. **Virtual assistants**: Once they have used a classification model to catch the *wake word*, and used ASR model to process your request, they can use a TTS model to respond to your inquiry.
3. **Navigation Systems**: Delivering turn-by-turn GPS directions and real-time traffic updates while driving
4. **Listen rather than read**: Converting novels, news articles, and blogs into audio formats for hands-free, on-the-go consumption.
      - *visually impaired users*
      - *reading difficulties*
      - *dyslexia*

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch

model = VitsModel.from_pretrained("wasmdashai/vits-ar-sa-huba-v2")
tokenizer = AutoTokenizer.from_pretrained("wasmdashai/vits-ar-sa-huba-v2")

In [20]:
from IPython.display import Audio

def generate_speech(text):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        full_generation = model(**inputs)
    full_generation_waveform = full_generation.waveform.cpu().numpy().reshape(-1)

    return Audio(full_generation_waveform, rate=model.config.sampling_rate)

In [21]:
generate_speech("السلام عليكم  كيفك  عساك بخير")

In [22]:
generate_speech("الخيل والليل والبيداء تعرفني")

In [23]:
generate_speech("الخَيْلُ واللَّيْلُ والبَيْداءُ تَعْرِفُنِي والسَّيْفُ والرُّمْحُ والقِرْطاسُ والقَلَمُ")

## 4. Voice Activity Detection (VAD)

Tells you where speech begins and ends in an audio.

![](https://github.com/HassanAlgoz/AAI/blob/main/content/W5/M1/assets/vad.png?raw=1)

See: [`voice-activity-detection`](https://huggingface.co/models?pipeline_tag=voice-activity-detection) on the Hub.

## 5. Speaker Diarization

Determining "who spoke when" by segmenting audio recordings according to distinct speaker identities—crucial for meetings, interviews, and call center analytics.

![](https://github.com/HassanAlgoz/AAI/blob/main/content/W5/M1/assets/speaker_diarization.png?raw=1)

Representative Models:

- **pyannote-audio** pipeline (pretrained on VoxCeleb and AMI datasets) is widely used for robust, plug-and-play diarization.
- **UIS-RNN** for online and streaming scenarios with incremental speaker assignment.

See: [`speaker-diarization`](https://huggingface.co/models?other=speaker-diarization) on the Hub.

## 6. Music-source Separation

Isolating specific audio tracks from a mixture or cleaning up noisy audio (e.g., separating vocals from instruments, removing background static).

![](https://github.com/HassanAlgoz/AAI/blob/main/content/W5/M1/assets/music_source_separation.png?raw=1)

Representative Models:

- **HTDemucs** for music stem separation.
- **SepFormer** for isolating clean speech from complex acoustic environments.

See: [`music-source-separation`](https://huggingface.co/models?other=music-source-separation&sort=trending) on the Hub.